# 07.2 - Model Optimization - Fine-Tuning

This notebook demonstrates supervised fine-tuning (SFT) of a small language model on a science Q&A dataset using Hugging Face's ecosystem:

- **`datasets`** — load and preprocess the training corpus
- **`transformers`** — load the base model and tokenizer
- **`peft`** — attach LoRA adapters for parameter-efficient fine-tuning
- **`bitsandbytes`** — quantize the frozen base weights to 4-bit (QLoRA) to reduce GPU memory
- **`trl`** — `SFTTrainer` orchestrates the training loop

The result is a LoRA adapter that can be merged back into the base model or served on its own. GPU with at least 8 GB VRAM is recommended.

## 00. Dependencies

Install the fine-tuning stack. `trl` and `peft` are not in the main project dependencies, so install them here if needed.

In [ ]:
%pip install -q trl peft accelerate datasets

## 01. Configuration

All run-level knobs live here. Change `base_model_id` to swap the backbone; adjust `lora_r` and `lora_alpha` to trade off adapter capacity vs. memory.

In [ ]:
# Base model — small enough to run on a single consumer GPU
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# Where to write the adapter and merged model
adapter_path = "../output/models/qwen2.5-0.5b-sciq-lora"
merged_model_path = "../output/models/qwen2.5-0.5b-sciq-merged"

# LoRA hyperparameters
lora_r = 16          # rank — higher = more capacity, more VRAM
lora_alpha = 32      # scaling factor; rule of thumb: 2 * lora_r
lora_dropout = 0.05

# Training hyperparameters
num_train_epochs = 3
per_device_train_batch_size = 4
gradient_accumulation_steps = 4  # effective batch = 4 * 4 = 16
learning_rate = 2e-4
max_seq_length = 512
dataset_subset_size = 2000       # use a subset of SciQ for a quick demo; set to None for full dataset

## 02. Dataset

[SciQ](https://huggingface.co/datasets/allenai/sciq) is a science exam question dataset with ~13 k multiple-choice questions drawn from Physics, Chemistry, Biology, and Earth Science. Each row contains:

| Field | Description |
|---|---|
| `question` | The question text |
| `correct_answer` | The ground-truth answer |
| `support` | A short passage supporting the correct answer |
| `distractor1/2/3` | Plausible wrong answers |

We will fine-tune the model to read the support passage and answer the question correctly — a reading-comprehension style task.

In [ ]:
from datasets import load_dataset

raw = load_dataset("allenai/sciq")
raw

In [ ]:
import pandas as pd

pd.DataFrame(raw["train"].select(range(5)))

### Format as chat messages

Convert each example into the chat template that `Qwen2.5-Instruct` expects. The system prompt tells the model its role; the user turn provides the passage and question; the assistant turn is the correct answer.

The tokenizer's `apply_chat_template` renders this into a single string that `SFTTrainer` can train on directly.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(base_model_id)


def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "You are a science tutor. Read the provided passage and answer the question concisely.",
        },
        {
            "role": "user",
            "content": f"Passage: {example['support']}\n\nQuestion: {example['question']}",
        },
        {
            "role": "assistant",
            "content": example["correct_answer"],
        },
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}


# Filter out examples with empty support passages, then format
train_ds = raw["train"].filter(lambda x: len(x["support"]) > 20)
val_ds = raw["validation"].filter(lambda x: len(x["support"]) > 20)

if dataset_subset_size:
    train_ds = train_ds.select(range(min(dataset_subset_size, len(train_ds))))

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
val_ds = val_ds.map(format_example, remove_columns=val_ds.column_names)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")
print("\n--- Sample formatted example ---")
print(train_ds[0]["text"])

## 03. Base Model (QLoRA)

Load the base model in 4-bit using `bitsandbytes` so the frozen weights occupy ~half the VRAM they would at `bfloat16`. Only the small LoRA adapter weights (added next) will be trained in full precision — this is the QLoRA technique.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4 quantization preserves more signal than fp4
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # second quantization of the quantization constants
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.config.use_cache = False  # required for gradient checkpointing
model.config.pretraining_tp = 1

print(f"Model dtype: {model.dtype}")
print(f"Loaded on: {next(model.parameters()).device}")

## 04. LoRA Adapter

LoRA (Low-Rank Adaptation) injects two small matrices, **A** (d × r) and **B** (r × d), into selected linear layers. During training only A and B are updated; the frozen base weights are unchanged. At inference the adaptation **BA** is added to the original weight.

`target_modules` lists the projection layers that receive adapters. For Qwen2 models these are the attention projections and the MLP gate/up/down projections.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare frozen quantized weights so gradients can flow through them into the adapters
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 05. Training

`SFTTrainer` (Supervised Fine-Tuning Trainer) from TRL wraps `Trainer` with sensible defaults for instruction-tuning. It reads the `"text"` column we created, tokenizes on the fly, and packs short sequences together when `packing=True` to improve GPU utilization.

Key training arguments:

| Argument | Value | Reason |
|---|---|---|
| `fp16` / `bf16` | auto-detected | mixed-precision forward pass |
| `gradient_checkpointing` | `True` | trade compute for memory |
| `optim` | `paged_adamw_32bit` | bitsandbytes paged optimizer keeps optimizer states in CPU RAM |
| `lr_scheduler_type` | `cosine` | smooth decay avoids late-training instability |

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir=adapter_path,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=learning_rate,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch",
    report_to="none",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
)

In [ ]:
trainer.train()

Plot training loss to verify the model is learning and not diverging.

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_logs = [e for e in log_history if "loss" in e and "eval_loss" not in e]
eval_logs = [e for e in log_history if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([e["step"] for e in train_logs], [e["loss"] for e in train_logs], label="train loss")
if eval_logs:
    ax.plot([e["step"] for e in eval_logs], [e["eval_loss"] for e in eval_logs], label="val loss", marker="o")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss")
ax.legend()
plt.tight_layout()
plt.show()

## 06. Save the LoRA Adapter

Saving with `save_pretrained` writes only the adapter weights and the PEFT config — a fraction of the full model size. This adapter can be loaded on top of any compatible base model without re-merging.

In [ ]:
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to: {adapter_path}")

## 07. Inference with the Fine-Tuned Adapter

Load the adapter on top of the base model and run a sample inference. The model is still in 4-bit here; for deployment you would merge and optionally re-quantize.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_tokenizer = AutoTokenizer.from_pretrained(adapter_path)

pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    max_new_tokens=64,
    do_sample=False,
)

In [ ]:
# Grab a held-out example from the validation set
sample = raw["test"][0]

messages = [
    {
        "role": "system",
        "content": "You are a science tutor. Read the provided passage and answer the question concisely.",
    },
    {
        "role": "user",
        "content": f"Passage: {sample['support']}\n\nQuestion: {sample['question']}",
    },
]

prompt = ft_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
result = pipe(prompt)[0]["generated_text"]
answer = result[len(prompt):].strip()

print(f"Question  : {sample['question']}")
print(f"Expected  : {sample['correct_answer']}")
print(f"Predicted : {answer}")

## 08. Merge LoRA into Base Weights

Merging folds the adapter matrices into the original weight matrices so the result is a standalone model with no PEFT dependency. This is the artifact to ship for production or upload to the Hub.

Merge requires loading the base model in **full precision** (not quantized) because the arithmetic is done in bfloat16. Expect higher peak memory usage during this step.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base in bf16 (no quantization) for a clean merge
base_fp = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

peft_model = PeftModel.from_pretrained(base_fp, adapter_path)
merged = peft_model.merge_and_unload()

merged.save_pretrained(merged_model_path, safe_serialization=True)
tokenizer.save_pretrained(merged_model_path)

print(f"Merged model saved to: {merged_model_path}")

## 09. (Optional) Push to Hugging Face Hub

Authenticate and upload the merged model so it can be loaded anywhere with `from_pretrained`. Requires an HF account and a token with write access.

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import os

# Replace with your HF username
hf_username = os.environ.get("HF_USERNAME", "your-hf-username")
hub_model_id = f"{hf_username}/qwen2.5-0.5b-sciq-lora"

merged.push_to_hub(hub_model_id, safe_serialization=True)
tokenizer.push_to_hub(hub_model_id)

print(f"Pushed to: https://huggingface.co/{hub_model_id}")